In [1]:
DOMAIN = "https://kortney-ganglionate-fatly.ngrok-free.dev"

### Install environment

In [2]:
import importlib.util
import subprocess
import sys


def run_pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", *args])


def run_cmd(cmd, shell: bool = False):
    subprocess.check_call(cmd, shell=shell)


if importlib.util.find_spec("flashrank") is None:
    run_pip("install", "vllm==0.10.0")
    run_pip("install", "triton==3.2.0")
    run_pip(
        "install",
        "flashrank",
        "langchain",
        "langchain-community",
        "langchain_google_genai",
        "openai",
        "faiss-cpu",
        "langchain_huggingface",
        "crawl4ai",
        "unidecode",
        "pymupdf4llm",
        "google-genai",
        "rapidfuzz",
        "transformers==4.57.0",
    )
    run_pip("uninstall", "-y", "openai")
    run_pip("install", "openai==1.90.0")
    run_cmd([
        "wget",
        "-q",
        "-O",
        "ngrok.zip",
        "https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.zip",
    ])
    run_cmd(["unzip", "-o", "ngrok.zip"])
    run_cmd(["mv", "ngrok", "/usr/local/bin/ngrok"])
    run_cmd(["chmod", "+x", "/usr/local/bin/ngrok"])
    run_cmd("ps aux | grep ngrok", shell=True)
else:
    print("All libs aldready installed")
import os
# Fixed, do not change
os.environ["GLOO_SOCKET_NAME"] = "eth0"
os.environ["NCCL_SOCKET_NAME"] = "eth0"
os.environ["VLLM_HOST_IP"] = "127.0.0.1" # Internal ip for data communicate between VLLM components
os.environ["VLLM_USE_V1"] = "0" # T4 have compute capacity of 7.5, it need at least 8.0 to use V1

All libs aldready installed


##### Download package from server

In [3]:
IS_LOCAL = DOMAIN == "http://127.0.0.1:8000"
BASE_PATH = "" if IS_LOCAL else "/kaggle/working/"

In [4]:
import requests
import io
import tarfile
import shutil
def unpack_folder(data: bytes, path: str):
    if os.path.exists(path): # Remove old code
        shutil.rmtree(path)
    with io.BytesIO(data) as tar_buffer:
        with tarfile.open(fileobj=tar_buffer, mode='r:gz') as tar:
            tar.extractall(path=path)
def unpack_file(data: bytes, path: str):
    os.makedirs(f"{BASE_PATH}files", exist_ok=True)
    if os.path.exists(path):
        os.remove(path)
    with open(f"{BASE_PATH}files/{path}", 'wb') as file:
        file.write(data)
def unpack_list(*names: str):
    # if DOMAIN == "http://127.0.0.1:8000": return
    for name in names:
        if "." in name:
            url = f"{DOMAIN}/package/{name}"
        else:
            url = f"{DOMAIN}/package/{name}"
        data = requests.get(url).content
        if "." in name:
            unpack_file(data, name)
        else:
            unpack_folder(data, name)
# if not os.path.exists(f"{BASE_PATH}/files"):
#     """"""
#     # This consume a lot of quota. So only download when necessary
# unpack_list(
#     "worker.env", "school_name.json", "school_alias.json", "local.pkl",
#     "data_retriever", "server", "instruction", "school_mapper", 
#     "lora/reader_v2", "lora/qwen_reranker_06b_v1"
# )

In [5]:
unpack_list("data_retriever")

In [6]:
# !pip install -U numpy==1.26.4 scipy==1.13.1 scikit-learn==1.5.1 \
# transformers==4.53.3 tokenizers==0.21.1 \
# --force-reinstall --no-cache-dir
# !pip install rapidfuzz
# !pip install pdfplumber

### Config

Load environment variable (api keys)

In [7]:
from dotenv import load_dotenv
load_dotenv(f"{BASE_PATH}files/worker.env")

True

Setup ngrok

In [8]:
NGROK_PORT = 8002
if DOMAIN != "http://127.0.0.1:8000":
    import subprocess
    subprocess.run(["ngrok", "config", "add-authtoken", os.getenv("NGROK_TOKEN_1", "")])
    subprocess.Popen(["ngrok", "http", str(NGROK_PORT)], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


Login hugging face

In [9]:
cmd = [
    "hf", "auth", "login",
    "--token", os.getenv("HUGGING_FACE_TOKEN")
]
import subprocess
subprocess.run(cmd)
print("")


The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: read).
The token `Nam Hai Nguyen` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `Nam Hai Nguyen`


##### Config

In [10]:
from data_retriever import *
from server import *
from school_mapper import SchoolMapper
from typing import AsyncGenerator, NotRequired, Protocol
from typing import Callable, AsyncGenerator
from openai import AsyncOpenAI, OpenAI
from google import genai
from google.genai import types
import os
import pickle
import json
import asyncio
import enum
import traceback
import copy
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from vllm.lora.request import LoRARequest

2025-12-02 08:24:05.698161: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764663845.720326     778 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764663845.727012     778 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Consider using the pymupdf_layout package for a greatly improved page layout analysis.


In [11]:
from typing import Protocol, AsyncGenerator, TypedDict
class KeywordInfo(TypedDict):
    query: str
    priority: float
    info: str
    school: str
class KeywordModelProtocol(Protocol):
    async def keywords(self, question: str, params: GenerationParams, threshold: float = 0.5) -> list[KeywordInfo]: ...
class RouterModelProtocol(Protocol):
    async def route(self, question: str, params: GenerationParams) -> list[dict]: ...

In [12]:
MODEL_ID = "Qwen/Qwen3-4B"
# Retriever config
search_config = WebsearchConfig(
    page_timeout=15,
    file_timeout=15,
)
rag_config = RagConfig(
    embedding_name="intfloat/multilingual-e5-small",
    device="cuda",
    use_hybrid=True,
    hybrid_alpha=0.7
)
splitter_config = SplitterConfig(
    tokenizer_name=MODEL_ID,
    chunk_size=512,
    chunk_overlap=64,
)
table_merge_config = MergeTableConfig(
    k_max_previous=5,
    k_max_next=5
)
neighbor_config = MergeNeighborConfig(
    k_previous_chunks=1,
    k_next_chunks=1
)
# Sampling Params
PAGE_RERANKER_PARAMS = {
    "temperature": 0.7,
    "top_p": 0.9,
    "max_tokens": 4096
}
KEYWORDS_PARAMS = {
    "temperature": 0.5,
    "top_p": 0.9,
    "max_tokens": 4096
}
ROUTER_PARAMS = {
    "temperature": 0.7,
    "top_p": 0.9,
    "max_tokens": 1024
}
SEP = "$$$"
MODELS: list[ModelInfo] = [
    {
        "name": "Qwen3 4B",
        "id": "Qwen/Qwen3-4B"
    },
    {
        "name": "Qwen3 4B LoRA",
        "id": f"Qwen/Qwen3-4B{SEP}1"
    }
]
CLIENT_INFO: WorkerServerInfo = {
    "name": "Test Qwen4B",
    "domain": "http://127.0.0.1:8002", # Auto change when run with ngrok
    "models": MODELS
}
READER_LORA = LoRARequest(
    lora_int_id= 1,
    lora_name= "Qwen Reader Lora",
    lora_path= f"{BASE_PATH}lora/reader_v2"
)
LORA_MAP = {
    1: READER_LORA
}

##### Template, instruction, prefix

In [13]:
from instruction import *

### Utility class

##### Data Retriever

In [14]:
class LocalRetriever:
    """Search in static db"""
    def __init__(self) -> None:
        with open(f"{BASE_PATH}files/local.pkl", 'rb') as file:
            self.all_docs = pickle.load(file)
    def _filter_docs(self, school_id: str, section: str) -> list:
        school_docs = [doc for doc in self.all_docs if doc.metadata.get("school_id") == school_id]
        if school_docs:
            filtered_docs = [doc for doc in school_docs if doc.metadata.get("section") == section]
        else:
            filtered_docs = []
        return filtered_docs
    def retrieve(self, keywords: list[dict]) -> tuple[list[WebSource], list[RagSource]]:
        web_sources: list[WebSource] = []
        rag_sources: list[RagSource] = []
        try:
            for kw in keywords:
                school_id = kw.get("school_id")
                section = kw.get("section")
                if school_id and section:
                    docs = self._filter_docs(school_id, section)
                    title = f"Tìm trường ĐH-CĐ - Cốc Cốc ({school_id})"
                    combined_content = "\n\n".join([doc.page_content for doc in docs])
                    description = combined_content[:100] + "..." if len(combined_content) > 100 else combined_content
                    web_source: WebSource = {
                        "query": f"{school_id}:{section}",
                        "title": title,
                        "description": description,
                        "url": "https://hoctap.coccoc.com/tim-truong-dh-cd",
                        "text": combined_content,
                        "files": [],
                        "score": 1
                    }
                    rag_source: RagSource = {
                        "chunk_index": 0,
                        "query": f"{school_id}:{section}",
                        "title": title,
                        "url": "https://hoctap.coccoc.com/tim-truong-dh-cd",
                        "text": combined_content,
                    }
                    web_sources.append(web_source)
                    rag_sources.append(rag_source)
        except:
            traceback.print_exc()
        finally:            
            return web_sources, rag_sources

In [15]:
class WebRetriever:
    """Search in web"""
    def __init__(self, llm_ranker: PageRerankModelProtocol, llm_keywords: KeywordModelProtocol) -> None:
        self.pipeline = DataRetrieverPipeline(
            llm_ranker,
            websearch_config=search_config,
            rag_config=rag_config,
            splitter_config=splitter_config,
            neighbor_merge_config=neighbor_config,
            table_merge_config=table_merge_config
        )
        self.llm_keywords = llm_keywords
        self.school_mapper = SchoolMapper(f"{BASE_PATH}files/school_name.json")
    async def start(self):
        """Initialize websearch"""
        await self.pipeline.start()
    async def retrive(self, question: str, params: GenerationParams) -> tuple[list[WebSource], list[RagSource]]:
        data = await self.llm_keywords.keywords(question, params)
        max_query = params.get("max_query", 1)
        queries = []
        school_restrict = params.get("school_domain", False)
        for item in data:
            if not school_restrict:
                queries.append(item["query"])
            else:
                school = item["school"]
                if school.strip() != "":
                    school_domains = self.school_mapper.domains_from_auto(school, 5)[:10]
                    print(f"[DOMAINS]", school_domains)
                    if len(school_domains) > 0:
                        queries.append([item["query"], school_domains])
        return await self.pipeline.retrieve(params, queries[:max_query])

In [16]:
class RouterRetriever:
    def __init__(self, llm_router: RouterModelProtocol, web_retriever: WebRetriever, local_retriever: LocalRetriever) -> None:
        self.web_retriever = web_retriever
        self.local_retriever = local_retriever
        self.router = llm_router
    async def retrieve(self, question: str, params: GenerationParams) -> tuple[list[WebSource], list[RagSource]]:
        use_websearch = params.get("use_websearch", False) and params.get("max_query", 0) > 0 and params.get("k_docs", 0) > 0 and params.get("k_pages", 0) > 0
        use_localdb = params.get("use_localdb", False)
        if use_websearch and use_localdb:
            local_queries = await self.router.route(question, params)
            if len(local_queries) > 0:
                return self.local_retriever.retrieve(local_queries)
            else:
                return await self.web_retriever.retrive(question, params)
        elif use_localdb:
            local_queries = await self.router.route(question, params)
            if len(local_queries) > 0:
                return self.local_retriever.retrieve(local_queries)
            else:
                return [], []
        elif use_websearch:
            return await self.web_retriever.retrive(question, params)
        else:
            return [], []

##### Client to call model

In [17]:
from vllm import SamplingParams, AsyncLLMEngine, AsyncEngineArgs
from vllm.outputs import RequestOutput
from vllm.utils import random_uuid
from vllm.lora.request import LoRARequest
from typing import AsyncGenerator
from typing import Optional, Any
from vllm.transformers_utils.tokenizers import MistralTokenizer
from openai.types.chat import  ChatCompletionUserMessageParam, ChatCompletionSystemMessageParam
from vllm.entrypoints.chat_utils import (
    ChatTemplateContentFormatOption, 
    resolve_chat_template_content_format, 
    apply_hf_chat_template,
    apply_mistral_chat_template,
    parse_chat_messages
)
from vllm.inputs.data import TokensPrompt

class AsyncLLMEngineWrapper:
    """Not support shutdown"""
    def __init__(self) -> None:
        self.engine = None
        self.reap_wait_time = 5
    def init(self, engine_args: AsyncEngineArgs):
        self.engine = AsyncLLMEngine.from_engine_args(engine_args)
    def generate(self, prompt: str | TokensPrompt, sampling_params: SamplingParams, lora_request: LoRARequest | None) -> AsyncGenerator[RequestOutput, None]:
        if self.engine is None:
            raise Exception("Not initialized")
        return self.engine.generate(
            prompt=prompt,
            sampling_params=sampling_params,
            request_id=random_uuid(),
            lora_request=lora_request
        )
    async def chat(self, instruction: str, prompt: str, sampling_params: SamplingParams, lora_request: LoRARequest | None = None):
        messages = [
            ChatCompletionSystemMessageParam(content=instruction, role="system"),
            ChatCompletionUserMessageParam(content=prompt, role="user")
        ]
        return await self._chat(
            messages=messages,
            sampling_params=sampling_params,
            lora_request=lora_request,
            chat_template_kwargs={
                "enable_thinking": False
            }
        )
    async def _chat(
        self,
        messages: list[ChatCompletionUserMessageParam | ChatCompletionUserMessageParam],
        sampling_params: SamplingParams,
        lora_request: LoRARequest | None,
        chat_template_content_format: ChatTemplateContentFormatOption = "auto",
        chat_template: Optional[str] = None,
        add_generation_prompt: bool = True,
        continue_final_message: bool = False,
        chat_template_kwargs: Optional[dict[str, Any]] = None
    ):
        if self.engine is None: raise Exception("Model not loaded")
        tokenizer = await self.engine.get_tokenizer(lora_request)
        model_config = self.engine.engine.get_model_config()
        resolved_content_format = resolve_chat_template_content_format(
            chat_template,
            None,
            chat_template_content_format,
            tokenizer,
            model_config=model_config,
        )
        _chat_template_kwargs: dict[str, Any] = dict(
            chat_template=chat_template,
            add_generation_prompt=add_generation_prompt,
            continue_final_message=continue_final_message,
            tools=None,
        )
        _chat_template_kwargs.update(chat_template_kwargs or {})
        conversation, _ = parse_chat_messages(
            messages, #type:ignore
            model_config,
            tokenizer,
            content_format=resolved_content_format,
        )

        if isinstance(tokenizer, MistralTokenizer):
            prompt_token_ids = apply_mistral_chat_template(
                tokenizer,
                messages=messages, #type:ignore
                **_chat_template_kwargs,
            )
        else:
            prompt_str = apply_hf_chat_template(
                tokenizer=tokenizer, #type:ignore
                conversation=conversation,
                model_config=model_config,
                **_chat_template_kwargs,
            )
            prompt_token_ids = tokenizer.encode(prompt_str, add_special_tokens=False)
        prompt = TokensPrompt(prompt_token_ids=prompt_token_ids)
        return self.generate(
            prompt,
            sampling_params=sampling_params,
            lora_request=lora_request,
        )

INFO 12-02 08:24:18 [__init__.py:235] Automatically detected platform cuda.


In [18]:
import msgspec
import glob

def _find_reranker_checkpoint(base_path: str = None) -> str | None:
    """Tìm LoRA adapter trong package/lora/qwen_reranker_06b_v1"""
    if base_path is None:
        base_path = BASE_PATH
    print(f"[Reranker] Searching for adapter, BASE_PATH={base_path}, cwd={os.getcwd()}")
    
    # Các đường dẫn có thể có (ưu tiên local trước)
    adapter_paths = [
        # Local paths (ưu tiên)
        "app/package/lora/qwen_reranker_06b_v1",
        os.path.join(os.getcwd(), "app", "package", "lora", "qwen_reranker_06b_v1"),
        # Downloaded paths
        f"{base_path}lora/qwen_reranker_06b_v1",
        f"{base_path}package/lora/qwen_reranker_06b_v1",
        "lora/qwen_reranker_06b_v1",
        "package/lora/qwen_reranker_06b_v1",
        # Other possible paths
        os.path.join(os.getcwd(), "package", "lora", "qwen_reranker_06b_v1"),
        os.path.join(os.getcwd(), "lora", "qwen_reranker_06b_v1"),
    ]
    
    for adapter_dir in adapter_paths:
        # Normalize path
        adapter_dir = os.path.normpath(adapter_dir)
        print(f"[Reranker] Checking: {adapter_dir} (exists: {os.path.exists(adapter_dir)})")
        if os.path.exists(adapter_dir) and os.path.isdir(adapter_dir):
            # Kiểm tra có adapter_model.safetensors
            adapter_file = os.path.join(adapter_dir, "adapter_model.safetensors")
            if os.path.exists(adapter_file):
                print(f"[Reranker] ✓ Found adapter at: {adapter_dir}")
                return adapter_dir  # Trả về thư mục chứa adapter
            else:
                print(f"[Reranker] Directory exists but adapter_model.safetensors not found")
                # List files in directory for debugging
                try:
                    files = os.listdir(adapter_dir)
                    print(f"[Reranker] Files in directory ({len(files)} files): {files}")
                except Exception as e:
                    print(f"[Reranker] Error listing directory: {e}")
        else:
            print(f"[Reranker] Path does not exist: {adapter_dir}")
    
    print("[Reranker] ✗ Adapter not found in any of the checked paths")
    return None

class VLLMModelCore:
    def __init__(self) -> None:
        self._engine = AsyncLLMEngineWrapper()
        self.logger = CmdLogger("Model")
        self._reranker_name = os.getenv("PAGE_RERANKER_MODEL", "Qwen/Qwen3-Reranker-0.6B")
        # Tìm checkpoint từ package/lora/qwen_reranker hoặc env variable
        self._reranker_checkpoint = os.getenv("PAGE_RERANKER_CHECKPOINT", None)
        if self._reranker_checkpoint is None:
            self._reranker_checkpoint = _find_reranker_checkpoint()
        if self._reranker_checkpoint:
            print(f"[Reranker] Found adapter at: {self._reranker_checkpoint}")
        else:
            print("[Reranker] No adapter checkpoint found, will use base model only")
        self._reranker_batch_size = int(os.getenv("PAGE_RERANKER_BATCH_SIZE", "8"))
        self._reranker_max_length = int(os.getenv("PAGE_RERANKER_MAX_LENGTH", "1024"))
        self._reranker_device = None
        self._reranker_model = None
        self._reranker_tokenizer = None
    def init(self, engine_args: AsyncEngineArgs):
        self._engine.init(engine_args)
    async def call(self, call_type: CallType, instruction: str, prompt: str, params: GenerationParams) -> AsyncGenerator[str, None]:
        print(f"[VLLM] {call_type} | Instruction length: {len(instruction)} | Prompt length: {len(prompt)} | kwargs: {params.get('kwargs')}")
        model_id = params["model_id"]
        lora_request = None
        if call_type == CallType.READER and SEP in model_id:
            lora_int_id = int(model_id.split(SEP)[-1])
            lora_request = LORA_MAP.get(lora_int_id)
        sampling_params = msgspec.convert(params, SamplingParams)
        if lora_request != None:
            print(f"[VLLm] Using {lora_request.lora_name}")
        stream = await self._engine.chat(
            instruction=instruction,
            prompt=prompt,
            sampling_params=sampling_params,
            lora_request=lora_request
        )
        total_text = ""
        last_index = 0
        async for event in stream:
            total_text = event.outputs[0].text
            yield total_text[last_index:]
            last_index = len(total_text)
    async def __call__(self, call_type: CallType, instruction: str, prompt: str, params: GenerationParams) -> AsyncGenerator[str, None]:
        return self.call(call_type, instruction, prompt, params)

In [19]:
class VLLMModel(VLLMModelCore):
    async def route(self, question: str, params: GenerationParams) -> list[dict]:
        text = ""
        prompt = ROUTER_TEMPLATE.format(question=question)
        copy_params = copy.deepcopy(params)
        copy_params.update(ROUTER_PARAMS) #type:ignore 
        async for chunk in await self(
            call_type=CallType.ROUTER, 
            instruction=ROUTER_INSTRUCTION, 
            prompt=ROUTER_PREFIX+prompt, 
            params=copy_params
        ):
            text += chunk
        try:
            self.logger.log(text)
            result = json.loads(extract_json(text))
            return result
        except:
            traceback.print_exc()
            return []
    def _ensure_reranker_loaded(self):
        if (
            self._reranker_model is not None
            and self._reranker_tokenizer is not None
            and self._reranker_device is not None
        ):
            return
        device_str = "cuda" if torch.cuda.is_available() else "cpu"
        self._reranker_device = torch.device(device_str)
        self._reranker_tokenizer = AutoTokenizer.from_pretrained(
            self._reranker_name,
            trust_remote_code=True
        )
        # Set padding token if not already set
        if self._reranker_tokenizer.pad_token is None:
            if self._reranker_tokenizer.eos_token is not None:
                self._reranker_tokenizer.pad_token = self._reranker_tokenizer.eos_token
            else:
                # Fallback: add a special pad token
                self._reranker_tokenizer.add_special_tokens({"pad_token": "[PAD]"})
        torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        self._reranker_model = AutoModelForSequenceClassification.from_pretrained(
            self._reranker_name,
            trust_remote_code=True,
            torch_dtype=torch_dtype
        )
        # Resize token embeddings if we added a new pad token
        if self._reranker_tokenizer.pad_token_id is not None:
            if self._reranker_model.config.pad_token_id is None:
                self._reranker_model.config.pad_token_id = self._reranker_tokenizer.pad_token_id
            # Resize embeddings if needed (only if we added a new token)
            if len(self._reranker_tokenizer) > self._reranker_model.get_input_embeddings().weight.shape[0]:
                self._reranker_model.resize_token_embeddings(len(self._reranker_tokenizer))
        self._reranker_model.to(self._reranker_device)
        
        # Load trained LoRA adapter or checkpoint if provided
        if self._reranker_checkpoint is not None and os.path.exists(self._reranker_checkpoint):
            try:
                print(f"[Reranker] Loading from {self._reranker_checkpoint}")
                
                # Check if it's a LoRA adapter directory
                if os.path.isdir(self._reranker_checkpoint):
                    adapter_file = os.path.join(self._reranker_checkpoint, "adapter_model.safetensors")
                    if os.path.exists(adapter_file):
                        # Load LoRA adapter using PEFT or direct loading
                        try:
                            from peft import PeftModel, PeftConfig
                            from safetensors.torch import load_file
                            import json
                            
                            # Check adapter config for num_labels BEFORE loading PEFT
                            adapter_config_file = os.path.join(self._reranker_checkpoint, "adapter_config.json")
                            checkpoint_num_labels = None
                            if os.path.exists(adapter_config_file):
                                try:
                                    with open(adapter_config_file, 'r') as f:
                                        adapter_config = json.load(f)
                                        # Check if num_labels is in the config
                                        if 'num_labels' in adapter_config:
                                            checkpoint_num_labels = adapter_config['num_labels']
                                            print(f"[Reranker] Found num_labels in adapter_config: {checkpoint_num_labels}")
                                except Exception as e:
                                    print(f"[Reranker] Could not read adapter_config.json: {e}")
                            
                            import json
                            
                            # Check adapter config for num_labels BEFORE loading PEFT
                            adapter_config_file = os.path.join(self._reranker_checkpoint, "adapter_config.json")
                            checkpoint_num_labels = None
                            if os.path.exists(adapter_config_file):
                                try:
                                    with open(adapter_config_file, 'r') as f:
                                        adapter_config = json.load(f)
                                        # Check if num_labels is in the config
                                        if 'num_labels' in adapter_config:
                                            checkpoint_num_labels = adapter_config['num_labels']
                                            print(f"[Reranker] Found num_labels in adapter_config: {checkpoint_num_labels}")
                                except Exception as e:
                                    print(f"[Reranker] Could not read adapter_config.json: {e}")
                            
                            # Load adapter state dict to check shapes - MUST be done first
                            print("[Reranker] Loading adapter state dict to check shapes...")
                            adapter_state_dict = load_file(adapter_file)
                            
                            # Detect num_labels from score.weight shape if not in config
                            if checkpoint_num_labels is None:
                                for k, v in adapter_state_dict.items():
                                    # Check for modules_to_save keys with score.weight
                                    if "score" in k and "weight" in k and "modules_to_save" in k:
                                        # Shape is [num_labels, hidden_size]
                                        checkpoint_num_labels = v.shape[0]
                                        print(f"[Reranker] Detected num_labels from {k} shape: {checkpoint_num_labels}")
                                        break
                                    elif k.endswith(".score.weight") or k == "score.weight":
                                        checkpoint_num_labels = v.shape[0]
                                        print(f"[Reranker] Detected num_labels from {k} shape: {checkpoint_num_labels}")
                                        break
                            
                            # Resize score layer if num_labels mismatch - MUST be done BEFORE PeftModel.from_pretrained
                            if checkpoint_num_labels is not None and checkpoint_num_labels != self._reranker_model.config.num_labels:
                                print(f"[Reranker] Resizing score layer: {self._reranker_model.config.num_labels} -> {checkpoint_num_labels}")
                                # Get hidden size
                                hidden_size = self._reranker_model.config.hidden_size
                                # Create new score layer with correct num_labels
                                import torch.nn as nn
                                new_score = nn.Linear(hidden_size, checkpoint_num_labels, bias=False)
                                # Copy existing weights if possible (take first num_labels if checkpoint has fewer)
                                if hasattr(self._reranker_model, 'score'):
                                    old_weight = self._reranker_model.score.weight
                                    if old_weight.shape[0] >= checkpoint_num_labels:
                                        new_score.weight.data = old_weight[:checkpoint_num_labels].clone()
                                    else:
                                        # Initialize new weights
                                        new_score.weight.data.normal_(mean=0.0, std=0.02)
                                # Replace score layer
                                self._reranker_model.score = new_score
                                self._reranker_model.config.num_labels = checkpoint_num_labels
                                self._reranker_model.num_labels = checkpoint_num_labels
                                self._reranker_model.to(self._reranker_device)
                                print(f"[Reranker] Score layer resized successfully to {checkpoint_num_labels} labels")
                            
                            adapter_state_dict = load_file(adapter_file)
                            
                            # Detect num_labels from score.weight shape if not in config
                            if checkpoint_num_labels is None:
                                for k, v in adapter_state_dict.items():
                                    # Check for modules_to_save keys with score.weight
                                    if "score" in k and "weight" in k and "modules_to_save" in k:
                                        # Shape is [num_labels, hidden_size]
                                        checkpoint_num_labels = v.shape[0]
                                        print(f"[Reranker] Detected num_labels from {k} shape: {checkpoint_num_labels}")
                                        break
                                    elif k.endswith(".score.weight") or k == "score.weight":
                                        checkpoint_num_labels = v.shape[0]
                                        print(f"[Reranker] Detected num_labels from {k} shape: {checkpoint_num_labels}")
                                        break
                            
                            # Resize score layer if num_labels mismatch - MUST be done BEFORE PeftModel.from_pretrained
                            if checkpoint_num_labels is not None and checkpoint_num_labels != self._reranker_model.config.num_labels:
                                print(f"[Reranker] Resizing score layer: {self._reranker_model.config.num_labels} -> {checkpoint_num_labels}")
                                # Get hidden size
                                hidden_size = self._reranker_model.config.hidden_size
                                # Create new score layer with correct num_labels
                                import torch.nn as nn
                                new_score = nn.Linear(hidden_size, checkpoint_num_labels, bias=False)
                                # Copy existing weights if possible (take first num_labels if checkpoint has fewer)
                                if hasattr(self._reranker_model, 'score'):
                                    old_weight = self._reranker_model.score.weight
                                    if old_weight.shape[0] >= checkpoint_num_labels:
                                        new_score.weight.data = old_weight[:checkpoint_num_labels].clone()
                                    else:
                                        # Initialize new weights
                                        new_score.weight.data.normal_(mean=0.0, std=0.02)
                                # Replace score layer
                                self._reranker_model.score = new_score
                                self._reranker_model.config.num_labels = checkpoint_num_labels
                                self._reranker_model.num_labels = checkpoint_num_labels
                                self._reranker_model.to(self._reranker_device)
                                print(f"[Reranker] Score layer resized successfully to {checkpoint_num_labels} labels")
                            
                            # Now load PEFT adapter - the model should have correct num_labels now
                            
                            # Check adapter config for num_labels BEFORE loading PEFT
                            adapter_config_file = os.path.join(self._reranker_checkpoint, "adapter_config.json")
                            checkpoint_num_labels = None
                            if os.path.exists(adapter_config_file):
                                try:
                                    with open(adapter_config_file, 'r') as f:
                                        adapter_config = json.load(f)
                                        if 'num_labels' in adapter_config:
                                            checkpoint_num_labels = adapter_config['num_labels']
                                            print(f"[Reranker] Found num_labels in adapter_config: {checkpoint_num_labels}")
                                except Exception as e:
                                    print(f"[Reranker] Could not read adapter_config.json: {e}")
                            
                            # Detect num_labels from score.weight shape if not in config
                            if checkpoint_num_labels is None:
                                for k, v in adapter_state_dict.items():
                                    if "score" in k and "weight" in k and "modules_to_save" in k:
                                        checkpoint_num_labels = v.shape[0]
                                        print(f"[Reranker] Detected num_labels from {k} shape: {checkpoint_num_labels}")
                                        break
                                    elif k.endswith(".score.weight") or k == "score.weight":
                                        checkpoint_num_labels = v.shape[0]
                                        print(f"[Reranker] Detected num_labels from {k} shape: {checkpoint_num_labels}")
                                        break
                            
                            # Resize score layer if num_labels mismatch - MUST be done BEFORE PeftModel.from_pretrained
                            if checkpoint_num_labels is not None and checkpoint_num_labels != self._reranker_model.config.num_labels:
                                print(f"[Reranker] Resizing score layer: {self._reranker_model.config.num_labels} -> {checkpoint_num_labels}")
                                hidden_size = self._reranker_model.config.hidden_size
                                import torch.nn as nn
                                new_score = nn.Linear(hidden_size, checkpoint_num_labels, bias=False)
                                if hasattr(self._reranker_model, 'score'):
                                    old_weight = self._reranker_model.score.weight
                                    if old_weight.shape[0] >= checkpoint_num_labels:
                                        new_score.weight.data = old_weight[:checkpoint_num_labels].clone()
                                    else:
                                        new_score.weight.data.normal_(mean=0.0, std=0.02)
                                self._reranker_model.score = new_score
                                self._reranker_model.config.num_labels = checkpoint_num_labels
                                self._reranker_model.num_labels = checkpoint_num_labels
                                self._reranker_model.to(self._reranker_device)
                                print(f"[Reranker] Score layer resized successfully to {checkpoint_num_labels} labels")
                            
                            # Extract modules_to_save weights (score.weight, classifier.weight) for later use
                            modules_to_save_weights = {}
                            model_dict = self._reranker_model.state_dict()
                            
                            for k, v in adapter_state_dict.items():
                                # Look for modules_to_save keys (score, classifier)
                                # They might be: score.weight, base_model.model.score.weight, etc.
                                model_key = None
                                if k == "score.weight" or k.endswith(".score.weight"):
                                    model_key = "score.weight"
                                elif k == "score.bias" or k.endswith(".score.bias"):
                                    model_key = "score.bias"
                                elif k == "classifier.weight" or k.endswith(".classifier.weight"):
                                    model_key = "classifier.weight"
                                elif k == "classifier.bias" or k.endswith(".classifier.bias"):
                                    model_key = "classifier.bias"
                                elif k.startswith("base_model.model.") and ("score" in k or "classifier" in k):
                                    model_key = k.replace("base_model.model.", "")
                                
                                if model_key and model_key in model_dict:
                                    # Handle shape mismatch by resizing if needed
                                    if model_dict[model_key].shape == v.shape:
                                        modules_to_save_weights[model_key] = v.to(self._reranker_device)
                                        print(f"[Reranker] Found modules_to_save: {k} -> {model_key} (shape: {v.shape})")
                            
                            # Load modules_to_save weights into model before PEFT
                            if modules_to_save_weights:
                                print(f"[Reranker] Loading {len(modules_to_save_weights)} modules_to_save weights into model...")
                                missing, unexpected = self._reranker_model.load_state_dict(modules_to_save_weights, strict=False)
                                if missing:
                                    print(f"[Reranker] Missing keys: {missing}")
                                if unexpected:
                                    print(f"[Reranker] Unexpected keys: {unexpected}")
                            
                            print("[Reranker] Loading LoRA adapter using PEFT")
                            self._reranker_model = PeftModel.from_pretrained(
                                self._reranker_model,
                                self._reranker_checkpoint,
                                device=self._reranker_device
                            )
                            
                            # Verify score.weight exists in adapter
                            if hasattr(self._reranker_model, 'peft_config'):
                                print(f"[Reranker] PEFT config loaded: {list(self._reranker_model.peft_config.keys())}")
                            
                            # Check if score.weight is in the model
                            model_state = self._reranker_model.state_dict()
                            score_keys = [k for k in model_state.keys() if 'score' in k]
                            print(f"[Reranker] Keys with 'score' in model: {score_keys}")
                            
                            # Merge LoRA weights into base model for faster inference
                            print("[Reranker] Merging LoRA weights into base model...")
                            self._reranker_model = self._reranker_model.merge_and_unload()
                            # Ensure model is on correct device after merge
                            self._reranker_model.to(self._reranker_device)
                            
                            # Verify score.weight after merge (merge_and_unload() should preserve modules_to_save weights)
                            final_state = self._reranker_model.state_dict()
                            score_weight_ok = False
                            if 'score.weight' in final_state:
                                print(f"[Reranker] ✓ score.weight present after merge (shape: {final_state['score.weight'].shape})")
                                # Check if it's not the default initialized weight
                                if hasattr(self._reranker_model, 'score'):
                                    score_weight = self._reranker_model.score.weight
                                    weight_norm = torch.norm(score_weight).item()
                                    print(f"[Reranker] score.weight norm: {weight_norm:.6f} (should be > 0 if loaded correctly)")
                                    if weight_norm >= 1e-6:
                                        score_weight_ok = True
                                    else:
                                        print(f"[Reranker] ⚠ WARNING: score.weight norm is very small, might be default initialized!")
                            else:
                                print(f"[Reranker] ✗ score.weight NOT found after merge")
                                print(f"[Reranker] Available keys with 'score': {[k for k in final_state.keys() if 'score' in k]}")
                            
                            # Only re-load modules_to_save weights if score.weight is missing or has invalid norm
                            if modules_to_save_weights and not score_weight_ok:
                                print("[Reranker] Re-loading modules_to_save weights after merge (score.weight was not preserved)...")
                                missing, unexpected = self._reranker_model.load_state_dict(modules_to_save_weights, strict=False)
                                # Filter out expected missing keys (base model weights)
                                if missing:
                                    base_model_keys = [k for k in missing if k.startswith('model.')]
                                    if len(base_model_keys) < len(missing):
                                        # Only print if there are unexpected missing keys
                                        unexpected_missing = [k for k in missing if not k.startswith('model.')]
                                        if unexpected_missing:
                                            print(f"[Reranker] Unexpected missing keys when re-loading: {unexpected_missing}")
                                if unexpected:
                                    print(f"[Reranker] Unexpected keys: {unexpected}")
                            elif modules_to_save_weights and score_weight_ok:
                                print("[Reranker] ✓ score.weight preserved after merge, no need to re-load")
                            
                            print(f"[Reranker] Successfully loaded and merged LoRA adapter from {self._reranker_checkpoint}")
                        except ImportError:
                            print("[Reranker] PEFT not available, loading adapter weights directly")
                            # Load directly from safetensors
                            from safetensors.torch import load_file
                            adapter_state_dict = load_file(adapter_file)
                            
                            print(f"[Reranker] Loaded {len(adapter_state_dict)} keys from adapter")
                            print(f"[Reranker] Sample adapter keys: {list(adapter_state_dict.keys())[:10]}")
                            
                            # Map adapter keys to model keys
                            model_dict = self._reranker_model.state_dict()
                            print(f"[Reranker] Model has {len(model_dict)} keys")
                            print(f"[Reranker] Model keys containing 'score': {[k for k in model_dict.keys() if 'score' in k]}")
                            
                            filtered_state_dict = {}
                            
                            for k, v in adapter_state_dict.items():
                                # LoRA adapter có thể có:
                                # 1. LoRA weights (lora_A, lora_B) - bỏ qua vì không có PEFT để merge
                                # 2. modules_to_save weights (score.weight, classifier.weight) - load trực tiếp
                                # 3. Keys với prefix base_model.model. - remove prefix
                                
                                model_key = k
                                if k.startswith("base_model.model."):
                                    model_key = k.replace("base_model.model.", "")
                                elif k.startswith("lora_"):
                                    # Skip LoRA weights khi không có PEFT
                                    continue
                                # Các keys khác (như score.weight, classifier.weight) giữ nguyên
                                
                                if model_key in model_dict:
                                    if model_dict[model_key].shape == v.shape:
                                        filtered_state_dict[model_key] = v.to(self._reranker_device)
                                        print(f"[Reranker] Loading key: {k} -> {model_key} (shape: {v.shape})")
                                    else:
                                        print(f"[Reranker] Shape mismatch for {k} -> {model_key}: adapter {v.shape} vs model {model_dict[model_key].shape}")
                                else:
                                    print(f"[Reranker] Key not in model: {k} -> {model_key}")
                            
                            print(f"[Reranker] Loading {len(filtered_state_dict)} matching keys into model")
                            missing_keys, unexpected_keys = self._reranker_model.load_state_dict(filtered_state_dict, strict=False)
                            
                            if missing_keys:
                                print(f"[Reranker] Missing keys after load: {missing_keys}")
                            if unexpected_keys:
                                print(f"[Reranker] Unexpected keys: {unexpected_keys}")
                            
                            # Verify score.weight was loaded
                            if 'score.weight' in filtered_state_dict:
                                print(f"[Reranker] ✓ score.weight successfully loaded (shape: {filtered_state_dict['score.weight'].shape})")
                            else:
                                print(f"[Reranker] ✗ score.weight NOT found in adapter or not loaded")
                                print(f"[Reranker] Available adapter keys with 'score': {[k for k in adapter_state_dict.keys() if 'score' in k]}")
                            
                            print(f"[Reranker] Successfully loaded adapter weights from {adapter_file}")
                            print(f"[Reranker] Note: LoRA weights not merged (PEFT not available). Only modules_to_save weights loaded.")
                    else:
                        print(f"[Reranker] adapter_model.safetensors not found in {self._reranker_checkpoint}")
                else:
                    # Handle regular checkpoint file
                    print(f"[Reranker] Loading checkpoint file")
                    
                    # Handle safetensors format
                    if self._reranker_checkpoint.endswith('.safetensors'):
                        try:
                            from safetensors.torch import load_file
                            state_dict = load_file(self._reranker_checkpoint)
                        except ImportError:
                            print("[Reranker] safetensors not installed, falling back to torch.load")
                            state_dict = torch.load(self._reranker_checkpoint, map_location=self._reranker_device)
                    else:
                        checkpoint = torch.load(self._reranker_checkpoint, map_location=self._reranker_device)
                        
                        # Handle different checkpoint formats
                        if isinstance(checkpoint, dict):
                            if 'model_state_dict' in checkpoint:
                                state_dict = checkpoint['model_state_dict']
                            elif 'state_dict' in checkpoint:
                                state_dict = checkpoint['state_dict']
                            else:
                                state_dict = checkpoint
                        else:
                            state_dict = checkpoint
                    
                    # Load state dict, handling mismatched keys
                    model_dict = self._reranker_model.state_dict()
                    
                    # Filter out keys that don't match
                    filtered_state_dict = {}
                    for k, v in state_dict.items():
                        if k in model_dict and model_dict[k].shape == v.shape:
                            filtered_state_dict[k] = v
                        else:
                            print(f"[Reranker] Skipping key {k} (shape mismatch or not in model)")
                    
                    # Load the filtered state dict
                    missing_keys, unexpected_keys = self._reranker_model.load_state_dict(filtered_state_dict, strict=False)
                    
                    if missing_keys:
                        print(f"[Reranker] Missing keys: {missing_keys}")
                    if unexpected_keys:
                        print(f"[Reranker] Unexpected keys: {unexpected_keys}")
                    
                    print(f"[Reranker] Successfully loaded checkpoint from {self._reranker_checkpoint}")
            except Exception as e:
                print(f"[Reranker] Failed to load checkpoint: {e}")
                traceback.print_exc()
        
        self._reranker_model.eval()
    def _make_page_text(self, page: SearchResult) -> str:
        parts = [
            page.get("title", ""),
            page.get("description", ""),
            page.get("url", "")
        ]
        return "\n\n".join([part for part in parts if part])
    async def _llm_rerank_page(self, pages: list[SearchResult], query: str, relative_threshold: float, params: GenerationParams) -> list[SearchResult]:
        if len(pages) == 0:
            return []
        try:
            self._ensure_reranker_loaded()
        except Exception as e:
            traceback.print_exc()
            raise RuntimeError(f"Failed to load reranker model: {e}") from e
        if (
            self._reranker_model is None
            or self._reranker_tokenizer is None
            or self._reranker_device is None
        ):
            raise RuntimeError("Reranker model, tokenizer, or device is not initialized")
        page_texts = [self._make_page_text(page) for page in pages]
        
        # DEBUG: Log reranker input
        print(f"\n[Reranker Debug] Query: {query}")
        print(f"[Reranker Debug] Number of pages: {len(pages)}")
        print(f"[Reranker Debug] Sample page texts (first 3):")
        for i, (page, page_text) in enumerate(zip(pages[:3], page_texts[:3]), 1):
            print(f"\n  Page {i}:")
            print(f"    Title: {page.get('title', 'N/A')[:100]}")
            print(f"    Description: {page.get('description', 'N/A')[:100]}")
            print(f"    URL: {page.get('url', 'N/A')}")
            print(f"    Page text (first 200 chars): {page_text[:200]}...")
        
        def _run_reranker() -> list[float]:
            scores: list[float] = []
            self._reranker_model.eval()
            for start in range(0, len(page_texts), self._reranker_batch_size):
                end = start + self._reranker_batch_size
                batch_texts = page_texts[start:end]
                batch_queries = [query for _ in batch_texts]
                
                # DEBUG: Log tokenizer input
                if start == 0:  # Only log first batch
                    print(f"\n[Reranker Debug] First batch:")
                    print(f"  Query: {batch_queries[0]}")
                    print(f"  First page text: {batch_texts[0][:300]}...")
                
                inputs = self._reranker_tokenizer(
                    batch_queries,
                    batch_texts,
                    padding=True,
                    truncation=True,
                    max_length=self._reranker_max_length,
                    return_tensors="pt"
                )
                inputs = {k: v.to(self._reranker_device) for k, v in inputs.items()}
                with torch.no_grad():
                    logits = self._reranker_model(**inputs).logits
                    logits = logits.view(-1)
                    batch_scores = logits.float().cpu().tolist()
                
                # DEBUG: Log raw scores
                if start == 0:  # Only log first batch
                    print(f"\n[Reranker Debug] Raw scores from first batch:")
                    for i, (score, page) in enumerate(zip(batch_scores, pages[start:end]), 1):
                        print(f"  [{i}] Score: {score:.6f} | Title: {page.get('title', 'N/A')[:80]}")
                
                scores.extend(batch_scores)
            return scores
        try:
            scores = await asyncio.to_thread(_run_reranker)
        except Exception as e:
            traceback.print_exc()
            raise RuntimeError(f"Failed to run reranker inference: {e}") from e
        if len(scores) != len(pages):
            raise RuntimeError(f"Reranker returned {len(scores)} scores but expected {len(pages)} scores")
        
        # DEBUG: Log all scores before threshold
        print(f"\n[Reranker Debug] All raw scores (before threshold):")
        score_page_pairs = [(float(score), page) for score, page in zip(scores, pages)]
        score_page_pairs.sort(key=lambda x: x[0], reverse=True)
        for i, (score, page) in enumerate(score_page_pairs, 1):
            print(f"  [{i}] Score: {score:.6f} | Title: {page.get('title', 'N/A')[:80]}")
        
        self.logger.log("-----Original-----")
        if self.logger._enable:
            for page in pages:
                self.logger.log(f'{page["score"]:.3f} + {page["title"]}')
        max_score = float("-inf")
        for score, search_result in zip(scores, pages):
            score = float(score)
            max_score = max(max_score, score)
            search_result["score"] = score
        if max_score == float("-inf"):
            raise RuntimeError("Reranker returned all invalid scores (all -inf)")
        threshold_score = max_score * relative_threshold
        
        # DEBUG: Log threshold info
        print(f"\n[Reranker Debug] Threshold calculation:")
        print(f"  Max score: {max_score:.6f}")
        print(f"  Relative threshold: {relative_threshold}")
        print(f"  Threshold score: {threshold_score:.6f}")
        
        results: list[SearchResult] = []
        for search_result in pages:
            if search_result["score"] >= threshold_score:
                results.append(search_result)
        results = sorted(results, key=lambda r: r["score"], reverse=True)
        
        # DEBUG: Log filtered results
        print(f"\n[Reranker Debug] Filtered results (after threshold, {len(results)}/{len(pages)}):")
        for i, page in enumerate(results, 1):
            print(f"  [{i}] Score: {page['score']:.6f} | Title: {page.get('title', 'N/A')[:80]}")
        
        self.logger.log("-----Reorder-----")
        if self.logger._enable:
            for page in results:
                self.logger.log(f'{page["score"]:.3f} + {page["title"]}')
        return results
    async def keywords(self, question: str, params: GenerationParams, threshold: float = 0.5) -> list[KeywordInfo]:
        num_queries = params.get("max_query", 1)
        copy_params = copy.deepcopy(params)
        copy_params.update(KEYWORDS_PARAMS) #type:ignore
        prompt = KEYWORD_TEMPLATE.format(question=question)
        text = ""
        async for chunk in await self(
            call_type=CallType.KEYWORDS, 
            instruction=KEYWORDS_INTRUCTION, 
            prompt=KEYWORDS_PREFIX.replace("{num}", str(num_queries))+prompt, 
            params=copy_params
        ):
            text += chunk
        try:
            self.logger.log(text)
            result: list[KeywordInfo] = json.loads(extract_json(text))
            for item in result:
                self.logger.log(item)
            return result
        except:
            print(text)
            traceback.print_exc()
            return []
    async def _heristic_rerank_page(self, pages: list[SearchResult], query: str, relative_threshold: float, params: GenerationParams) -> list[SearchResult]:
        """Rerank search results using embedding similarity"""
        self.logger.log("-----Original-----")
        if self.logger._enable:
            for page in pages:
                self.logger.log(f'{page["score"]:.3f} + {page["title"]}')
        import numpy as np
        def normalize_text(text: str) -> str:
            text = text.lower().strip()
            text = re.sub(r"[^a-zA-Z0-9\u00C0-\u1EF9\s\.,;]", " ", text)
            text = re.sub(r"\s+", " ", text).strip()
            return text
        def detect_school(query: str, schools: dict) -> str | None:
            """Detect school from query using predefined keywords"""
            for school, aliases in schools.items():
                if any(alias in query for alias in aliases):
                    return school
            return None
        schools = {
            school: [normalize_text(alias) for alias in aliases]
            for school, aliases in json.load(open(f"{BASE_PATH}files/school_alias.json", "r", encoding="utf-8")).items()
        }
        embedding = ws_pipeline.retriever.web_retriever.pipeline._rag.embedding
        
        # If no embedding model available, return original results
        if not embedding:
            return pages
        
        query_norm = normalize_text(query)
        detected_school = detect_school(query_norm, schools)
        
        try:
            query_emb = embedding.embed_query(query_norm)
            max_score = 0
            for page in pages:
                title = page.get("title", "") or ""
                desc = page.get("description", "") or ""
                url = page.get("url", "") or ""

                # Chuẩn hóa
                title_norm = normalize_text(title)
                desc_norm = normalize_text(desc)
                url_norm = normalize_text(url)

                # Semantic embedding
                title_emb = embedding.embed_query(title_norm) if title_norm else None
                desc_emb = embedding.embed_query(desc_norm) if desc_norm else None
                url_emb = embedding.embed_query(url_norm) if url_norm else None

                # Cosine similarity
                def cos_sim(a, b):
                    norm_a = np.linalg.norm(a)
                    norm_b = np.linalg.norm(b)
                    if norm_a == 0 or norm_b == 0:
                        return 0.0
                    return float(np.dot(a, b) / (norm_a * norm_b))

                score = 0.0
                weights = {"title": 0.5, "desc": 0.3, "url": 0.2}
                if title_emb is not None:
                    score += cos_sim(query_emb, title_emb) * weights["title"]
                if desc_emb is not None:
                    score += cos_sim(query_emb, desc_emb) * weights["desc"]
                if url_emb is not None:
                    score += cos_sim(query_emb, url_emb) * weights["url"]

                # Heuristic ưu tiên trường trong query
                if detected_school:
                    aliases = [normalize_text(a) for a in schools.get(detected_school, [])]
                    if any(a in text for a in aliases for text in [url_norm, title_norm, desc_norm]):
                        score += 0.5
                    else:
                        for school, other_aliases in schools.items():
                            if school != detected_school:
                                other_aliases_norm = [normalize_text(a) for a in other_aliases]
                                if any(a in text for a in other_aliases_norm for text in [url_norm, title_norm, desc_norm]):
                                    score -= 0.5

                # Heuristic boost
                if any(kw in query_norm for kw in ["tuyển sinh", "ngành đào tạo"]):
                    if "tuyensinh247" in url_norm:
                        score += 0.1
                    if url_norm.endswith(".edu") or ".edu.vn" in url_norm:
                        score += 0.2
                page["score"] = score
                max_score = max(score, max_score)
            threshold_score = max_score * relative_threshold
            # Sort theo score giảm dần
            results = []
            for page in pages:
                if page["score"] >= threshold_score:
                    results.append(page)
            results = sorted(results, key=lambda x: x["score"], reverse=True)
            self.logger.log("-----Reorder-----")
            if self.logger._enable:
                for page in results:
                    self.logger.log(f'{page["score"]:.3f} + {page["title"]}')
            return results
            
        except Exception:
            # If any error occurs, return original results
            return pages
    async def rerank_page(self, pages: list[SearchResult], query: str, relative_threshold: float, params: GenerationParams) -> list[SearchResult]:
        use_llm_rerank = params.get("llm_rerank", False)
        if use_llm_rerank:
            return await self._llm_rerank_page(pages, query, relative_threshold, params)
        else:
            return await self._heristic_rerank_page(pages, query, relative_threshold, params)

##### Pipeline

In [20]:
class CombinedProtocol(ModelProtocol, KeywordModelProtocol, PageRerankModelProtocol, RouterModelProtocol):
    pass
class CustomQA:
    def __init__(self, model_protocol: CombinedProtocol) -> None:
        self.logger = CmdLogger("QA")
        web_retriever = WebRetriever(model_protocol, model_protocol)
        local_retriever = LocalRetriever()
        self.retriever = RouterRetriever(
            model_protocol,
            web_retriever,
            local_retriever
        )
        self.llm_call = model_protocol
    async def start(self):
        await self.retriever.web_retriever.start()
    async def inference(self, prompt: str, request: WorkerChatRequest) -> AsyncGenerator[str, None]:
        text = ""
        async for chunk in await self.llm_call(
            call_type=CallType.READER, 
            instruction=READER_INSTRUCTION, 
            prompt=prompt, 
            params=request["params"]
        ):
            text += chunk
            yield chunk
    async def pre_inference(
        self,
        question: str,
        stream_id: str,
        params: GenerationParams
    ) -> tuple[str, ModelPreOutput]:
        web_sources, rag_sources = await self.retriever.retrieve(
            question, 
            params
        )
        print("\n" + "=" * 80)
        print(f"[RAG CHUNKS] Total {len(rag_sources)} chunks selected for reader:")
        for idx, chunk in enumerate(rag_sources, 1):
            title = chunk.get("title", "N/A")
            url = chunk.get("url", "N/A")
            chunk_idx = chunk.get("chunk_index", "N/A")
            text_preview = chunk.get("text", "")[:400]
            print("-" * 80)
            print(f"[Chunk {idx}] title={title} | url={url} | chunk_index={chunk_idx}")
            print("Text preview:\n" + text_preview)
        if not rag_sources:
            print("[RAG CHUNKS] No chunks selected.")
        print("=" * 80)
        context = SourceFormat()(rag_sources)
        prompt = READER_TEMPLATE.format(context=context, question=question)
        print("\n" + "=" * 80)
        print("[RAG CONTEXT] Formatted chunks sent to reader:")
        print("-" * 80)
        print(context if context.strip() else "[Empty context]")
        print("=" * 80)
        print("[FINAL PROMPT] Qwen4B input:")
        print("-" * 80)
        print(prompt)
        print("=" * 80 + "\n")
        self.logger.start()
        pre_output: ModelPreOutput = {
            "generation_params": params,
            "web_sources": web_sources,
            "rag_sources": rag_sources,
            "extra_data": {
            },
            "result_url": stream_id,
        }
        return prompt, pre_output

### Final

In [21]:
engine_args = AsyncEngineArgs(
    model=MODEL_ID,
    tensor_parallel_size=2,
    gpu_memory_utilization=0.7,
    max_model_len=32768,
    enable_lora=True,
    max_lora_rank=16,
    max_loras=1
)
vllm_model = VLLMModel()
vllm_model.init(engine_args)

[Reranker] Searching for adapter, BASE_PATH=/kaggle/working/, cwd=/kaggle/working
[Reranker] Checking: app/package/lora/qwen_reranker_06b_v1 (exists: False)
[Reranker] Path does not exist: app/package/lora/qwen_reranker_06b_v1
[Reranker] Checking: /kaggle/working/app/package/lora/qwen_reranker_06b_v1 (exists: False)
[Reranker] Path does not exist: /kaggle/working/app/package/lora/qwen_reranker_06b_v1
[Reranker] Checking: /kaggle/working/lora/qwen_reranker_06b_v1 (exists: True)
[Reranker] ✓ Found adapter at: /kaggle/working/lora/qwen_reranker_06b_v1
[Reranker] Found adapter at: /kaggle/working/lora/qwen_reranker_06b_v1
WARNING 12-02 08:24:35 [config.py:3392] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 12-02 08:24:35 [config.py:3443] Casting torch.bfloat16 to torch.float16.
INFO 12-02 08:24:35 [config.py:1604] Using max model len 32768
INFO 12-02 08:24:35 [llm_engine.py:228] Initializing a V

2025-12-02 08:24:40.582911: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764663880.602958     844 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764663880.608967     844 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


INFO 12-02 08:24:45 [__init__.py:235] Automatically detected platform cuda.
(VllmWorkerProcess pid=844) INFO 12-02 08:24:46 [multiproc_worker_utils.py:226] Worker ready; awaiting tasks
(VllmWorkerProcess pid=844) INFO 12-02 08:24:47 [cuda.py:346] Cannot use FlashAttention-2 backend for Volta and Turing GPUs.
(VllmWorkerProcess pid=844) INFO 12-02 08:24:47 [cuda.py:395] Using XFormers backend.
INFO 12-02 08:24:48 [__init__.py:1375] Found nccl from library libnccl.so.2
INFO 12-02 08:24:48 [pynccl.py:70] vLLM is using nccl==2.26.2
(VllmWorkerProcess pid=844) INFO 12-02 08:24:48 [__init__.py:1375] Found nccl from library libnccl.so.2
(VllmWorkerProcess pid=844) INFO 12-02 08:24:48 [pynccl.py:70] vLLM is using nccl==2.26.2
INFO 12-02 08:24:48 [custom_all_reduce_utils.py:246] reading GPU P2P access cache from /root/.cache/vllm/gpu_p2p_access_cache_for_0,1.json
(VllmWorkerProcess pid=844) INFO 12-02 08:24:48 [custom_all_reduce_utils.py:246] reading GPU P2P access cache from /root/.cache/vllm/

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


(VllmWorkerProcess pid=844) INFO 12-02 08:25:06 [default_loader.py:262] Loading weights took 16.29 seconds
(VllmWorkerProcess pid=844) INFO 12-02 08:25:06 [punica_selector.py:19] Using PunicaWrapperGPU.
(VllmWorkerProcess pid=844) INFO 12-02 08:25:06 [model_runner.py:1115] Model loading took 3.8589 GiB and 16.852403 seconds
INFO 12-02 08:25:06 [default_loader.py:262] Loading weights took 17.07 seconds
INFO 12-02 08:25:06 [punica_selector.py:19] Using PunicaWrapperGPU.
INFO 12-02 08:25:08 [model_runner.py:1115] Model loading took 3.8589 GiB and 17.546928 seconds
(VllmWorkerProcess pid=844) INFO 12-02 08:25:20 [worker.py:295] Memory profiling takes 11.84 seconds
(VllmWorkerProcess pid=844) INFO 12-02 08:25:20 [worker.py:295] the current vLLM instance can use total_gpu_memory (14.74GiB) x gpu_memory_utilization (0.70) = 10.32GiB
(VllmWorkerProcess pid=844) INFO 12-02 08:25:20 [worker.py:295] model weights take 3.86GiB; non_torch_memory takes 0.12GiB; PyTorch activation peak memory takes 1

Capturing CUDA graph shapes:   0%|          | 0/35 [00:00<?, ?it/s]

(VllmWorkerProcess pid=844) INFO 12-02 08:25:59 [custom_all_reduce.py:196] Registering 2555 cuda graph addresses
INFO 12-02 08:26:14 [custom_all_reduce.py:196] Registering 2555 cuda graph addresses
(VllmWorkerProcess pid=844) INFO 12-02 08:26:14 [model_runner.py:1537] Graph capturing finished in 50 secs, took 0.39 GiB
INFO 12-02 08:26:14 [model_runner.py:1537] Graph capturing finished in 50 secs, took 0.39 GiB
INFO 12-02 08:26:14 [llm_engine.py:424] init engine (profile, create kv cache, warmup model) took 66.36 seconds


Server wrapper

In [22]:
ws_pipeline = CustomQA(vllm_model)
await ws_pipeline.start()
import uuid
class ServerModelImplement(ServerModel):  
    def __init__(self) -> None:
        self.request_storage: dict[str, tuple[str, WorkerChatRequest, ModelPreOutput]] = {}
    async def pre_inference(self, request: WorkerChatRequest) -> ModelPreOutput:
        stream_id = str(uuid.uuid4())
        params = request["params"]
        print(params)
        prompt, pre_output = await ws_pipeline.pre_inference(
            request["text"],
            stream_id,
            request["params"]
        ) 
        self.request_storage[stream_id] = (prompt, request, pre_output)
        return pre_output
    async def inference(self, stream_id: str) -> AsyncGenerator[str, None]:
        prompt, request, pre_output = self.request_storage.pop(stream_id)
        generator = ws_pipeline.inference(prompt, request)
        total = ""
        try:
            async for chunk in generator:
                total += chunk
                yield chunk
        finally:
            # Store chat data when finish
            model_output: ModelOutput = {
                **pre_output,
                "text": total
            }
            data: WorkerStoreChatData = {
                "forward_kwargs": request["forward_kwargs"],
                "model_output": model_output
            }
            await self.store(data)

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Connect to server

In [ ]:
server_model = ServerModelImplement()
app = construct_app(
    server_domain=DOMAIN,
    info=CLIENT_INFO,
    server_model=server_model,
    init_tasks=[],
    shutdown_tasks=[],
    is_local=IS_LOCAL
)
# CORS policy
from fastapi.middleware.cors import CORSMiddleware
origins = [
    "http://127.0.0.1:8000", # I don't know why, but this won't work if not add this while in kaggle
    DOMAIN
]
app.add_middleware(
    CORSMiddleware,
    allow_origins=origins,
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"]
)
import uvicorn

uvicorn_config = uvicorn.Config(app, port=NGROK_PORT)
uvicorn_server = uvicorn.Server(uvicorn_config)
await uvicorn_server.serve()

INFO:     Started server process [778]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8002 (Press CTRL+C to quit)


Domain: https://cymosely-interventional-richie.ngrok-free.dev
{'model_id': 'Qwen/Qwen3-4B$$$1', 'use_websearch': True, 'use_localdb': False, 'max_query': 3, 'query_score_threshold': 0.5, 'engine_type': 'google', 'domain_restrict': False, 'school_domain': False, 'llm_rerank': True, 'page_score_threshold': 0.5, 'chunk_score_threshold': 0.5, 'k_docs': 5, 'k_pages': 3, 'page_rerank': False, 'chunk_rerank': False, 'include_pdf': False, 'include_image': False, 'merge_table': True, 'merge_neighbor': True, 'max_tokens': 2048, 'temperature': 0.5, 'top_p': 0.9, 'top_k': 40, 'max_history': 8}
[VLLM] CallType.KEYWORDS | Instruction length: 168 | Prompt length: 8669 | kwargs: None
INFO 12-02 08:26:58 [chat_utils.py:473] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.
INFO 12-02 08:26:58 [async_llm_engine.py:209] Added request f39a043ca244442cb01c1e35776c5d93.
INFO 12-02 08:26:59 [metrics.py:386] Avg prompt throughput: 65.0 tok

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.11/dist-packages/peft/config.py:165: UserWarning: Unexpected keyword arguments ['target_parameters'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


[Reranker] PEFT config loaded: ['default']
[Reranker] Keys with 'score' in model: ['base_model.model.score.original_module.weight', 'base_model.model.score.modules_to_save.default.weight']
[Reranker] Merging LoRA weights into base model...
[Reranker] ✓ score.weight present after merge (shape: torch.Size([1, 1024]))
[Reranker] score.weight norm: 0.647461 (should be > 0 if loaded correctly)
[Reranker] ✓ score.weight preserved after merge, no need to re-load
[Reranker] Successfully loaded and merged LoRA adapter from /kaggle/working/lora/qwen_reranker_06b_v1

[Reranker Debug] Query: điểm chuẩn trường đại học công nghệ đại học quốc gia hà nội 2025
[Reranker Debug] Number of pages: 10
[Reranker Debug] Sample page texts (first 3):

  Page 1:
    Title: Điểm chuẩn Trường Đại học Công nghệ, ĐHQG Hà Nội năm 2025
    Description: Aug 25, 2025 ... (Chinhphu.vn) - Lịch nghỉ tết Âm lịch, nghỉ lễ Quốc khánh năm 2026 cán bộ, công chứ
    URL: https://xaydungchinhsach.chinhphu.vn/diem-chuan-truong-dai

INFO:     Shutting down


(VllmWorkerProcess pid=844) INFO 12-02 08:33:17 [multiproc_worker_utils.py:260] Worker exiting
INFO 12-02 08:33:18 [multiproc_worker_utils.py:125] Killing local vLLM worker processes


ERROR:asyncio:Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x7d6ebc123bd0>


In [ ]:
DOMAIN

## Test WebSearch - Debug Tool

Test để xem:
1. Queries được extract từ câu hỏi
2. Web search results (sau reranker 0.6B)
3. Text content đọc từ các web


In [ ]:
async def test_websearch_debug(question: str, params: GenerationParams):
    """
    Test websearch: hiển thị queries, search results, và text content
    """
    print("=" * 80)
    print("WEBSEARCH DEBUG TEST")
    print("=" * 80)
    
    # Step 1: Extract keywords/queries
    print("\n📝 STEP 1: KEYWORD EXTRACTION")
    print("-" * 80)
    print(f"Question: {question}\n")
    
    keywords_data = await vllm_model.keywords(question, params)
    print(f"Extracted {len(keywords_data)} keywords/queries:")
    for i, item in enumerate(keywords_data, 1):
        print(f"\n  [{i}]")
        print(f"    Query: {item.get('query', 'N/A')}")
        print(f"    Priority: {item.get('priority', 'N/A')}")
        print(f"    Info: {item.get('info', 'N/A')}")
        print(f"    School: {item.get('school', 'N/A')}")
    
    # Step 2: Build queries list
    print("\n🔍 STEP 2: QUERY CONSTRUCTION")
    print("-" * 80)
    max_query = params.get("max_query", 1)
    queries = []
    school_restrict = params.get("school_domain", False)
    web_retriever = ws_pipeline.retriever.web_retriever
    
    for item in keywords_data:
        if not school_restrict:
            queries.append(item["query"])
        else:
            school = item["school"]
            if school.strip() != "":
                school_domains = web_retriever.school_mapper.domains_from_auto(school, 5)[:10]
                print(f"  School: {school} -> Domains: {school_domains}")
                if len(school_domains) > 0:
                    queries.append([item["query"], school_domains])
    
    queries = queries[:max_query]
    print(f"\nFinal queries (max_query={max_query}):")
    for i, query in enumerate(queries, 1):
        if isinstance(query, str):
            print(f"  [{i}] {query}")
        else:
            print(f"  [{i}] Query: {query[0]}")
            print(f"      Domains: {query[1]}")
    
    # Step 3: Search and retrieve
    print("\n🌐 STEP 3: WEB SEARCH & RETRIEVAL")
    print("-" * 80)
    web_sources, rag_sources = await web_retriever.retrive(question, params)
    
    print(f"\n✅ Web Sources: {len(web_sources)} (3 page đầu tiên từ search, KHÔNG qua reranker)")
    for i, ws in enumerate(web_sources, 1):
        print(f"\n  [{i}] {ws.get('title', 'N/A')}")
        print(f"      URL: {ws.get('url', 'N/A')}")
        print(f"      Score: {ws.get('score', 'N/A')}")
        print(f"      Description: {ws.get('description', 'N/A')[:200]}...")
        text = ws.get('text', '')
        print(f"      Text length: {len(text)} characters")
        preview = text[:500] + "..." if len(text) > 500 else text
        print(f"      Text preview (first 500 chars):\n      {preview}")
    
    print(f"\n✅ RAG Sources (chunks): {len(rag_sources)}")
    for i, rs in enumerate(rag_sources[:10], 1):  # Show first 10
        print(f"\n  [{i}] Chunk {rs.get('chunk_index', 'N/A')}")
        print(f"      URL: {rs.get('url', 'N/A')}")
        print(f"      Title: {rs.get('title', 'N/A')}")
        text = rs.get('text', '')
        preview = text[:400] + "..." if len(text) > 400 else text
        print(f"      Text (first 400 chars):\n      {preview}")
    
    if len(rag_sources) > 10:
        print(f"\n  ... and {len(rag_sources) - 10} more chunks")
    
    # Step 4: Show FULL TEXT of RAG chunks
    print("\n📚 STEP 4: FULL TEXT FROM RAG CHUNKS")
    print("-" * 80)
    for i, rs in enumerate(rag_sources, 1):
        print(f"\n{'='*80}")
        print(f"RAG CHUNK {i}/{len(rag_sources)}")
        print(f"{'='*80}")
        print(f"Chunk Index: {rs.get('chunk_index', 'N/A')}")
        print(f"URL: {rs.get('url', 'N/A')}")
        print(f"Title: {rs.get('title', 'N/A')}")
        text = rs.get('text', '')
        print(f"\nFull Text ({len(text)} characters):")
        print("-" * 80)
        print(text)
        print("-" * 80)
    
    # Step 5: Evaluate RAG chunks quality
    print("\n🔍 STEP 5: EVALUATE RAG CHUNKS QUALITY")
    print("-" * 80)
    # Extract keywords from question
    question_lower = question.lower()
    question_words = set(question_lower.split())
    # Also check for key phrases
    key_phrases = []
    for item in keywords_data:
        query = item.get('query', '').lower()
        key_phrases.extend(query.split())
    key_phrases = set(key_phrases)
    
    print(f"Question: {question}")
    print(f"Key words/phrases to check: {key_phrases}")
    print()
    
    relevant_chunks = []
    for i, rs in enumerate(rag_sources, 1):
        text = rs.get('text', '').lower()
        title = rs.get('title', '').lower()
        full_text = title + " " + text
        
        # Count matches
        matches = []
        for phrase in key_phrases:
            if phrase in full_text:
                matches.append(phrase)
        
        match_ratio = len(matches) / len(key_phrases) if len(key_phrases) > 0 else 0
        is_relevant = match_ratio >= 0.3  # At least 30% of keywords match
        
        status = "✅ RELEVANT" if is_relevant else "❌ NOT RELEVANT"
        print(f"[{i}] Chunk {rs.get('chunk_index', 'N/A')} - {status}")
        print(f"    Title: {rs.get('title', 'N/A')[:80]}")
        print(f"    Matched keywords: {matches} ({len(matches)}/{len(key_phrases)})")
        print(f"    Match ratio: {match_ratio:.2%}")
        
        if is_relevant:
            relevant_chunks.append(i)
    
    print(f"\n📊 Summary:")
    print(f"  Total chunks: {len(rag_sources)}")
    print(f"  Relevant chunks: {len(relevant_chunks)} ({len(relevant_chunks)/len(rag_sources)*100:.1f}%)")
    print(f"  Relevant chunk indices: {relevant_chunks}")
    
    # Step 6: Show full text from web sources
    print("\n📄 STEP 6: FULL TEXT FROM WEB SOURCES")
    print("-" * 80)
    for i, ws in enumerate(web_sources, 1):
        print(f"\n{'='*80}")
        print(f"WEB SOURCE {i}/{len(web_sources)}")
        print(f"{'='*80}")
        print(f"Title: {ws.get('title', 'N/A')}")
        print(f"URL: {ws.get('url', 'N/A')}")
        print(f"Score: {ws.get('score', 'N/A')}")
        print(f"\nFull Text ({len(ws.get('text', ''))} characters):")
        print("-" * 80)
        print(ws.get('text', 'N/A'))
        print("-" * 80)
    
    print("\n" + "=" * 80)
    print("✅ TEST COMPLETED")
    print("=" * 80)
    
    return {
        "keywords": keywords_data,
        "queries": queries,
        "web_sources": web_sources,
        "rag_sources": rag_sources
    }


### Run Test

Nhập câu hỏi và chạy test:


In [ ]:
# Test question
question = "Thủ khoa ngành trí tuệ nhân tạo UET 2025 là ai"

# Generation params - TẮT reranker, tải 3 page đầu tiên
params: GenerationParams = {
    "model_id": f"{MODEL_ID}{SEP}1",  # Use LoRA model
    "use_websearch": True,
    "use_localdb": False,  # Chỉ test websearch
    "max_query": 2,
    "k_docs": 5,
    "k_pages": 3,  # Tải 3 page đầu tiên
    "engine_type": "goggle",  # or "google"
    "domain_restrict": False,
    "school_domain": False,
    "llm_rerank": False,  # TẮT reranker - lấy 3 page đầu tiên từ search
    "page_score_threshold": 0.5,
    "chunk_score_threshold": 0.5,
    "max_tokens": 2048,
    "temperature": 0.5,
    "top_p": 0.9,
}

print(f"Question: {question}")
print(f"Params: {params}")
print(f"\n⚠️ Reranker: TẮT - Sẽ tải 3 page đầu tiên từ search results")


In [ ]:
# Run test
result = await test_websearch_debug(question, params)


In [ ]:
# ================================================================================
# CHATBOT HOÀN CHỈNH: HỎI → WEBSEARCH → TRẢ LỜI
# ================================================================================

# Nhập câu hỏi của bạn
user_question = "Thủ khoa ngành trí tuệ nhân tạo UET 2025"

# Cấu hình generation params
generation_params: GenerationParams = {
    "model_id": f"{MODEL_ID}{SEP}1",  # Qwen 4B LoRA
    "use_websearch": True,
    "use_localdb": False,
    "max_query": 2,
    "k_docs": 5,
    "use_snippet_optimization": True,
    "k_pages": 5,
    "engine_type": "google",
    "domain_restrict": False,
    "school_domain": False,
    "llm_rerank": False,
    "page_score_threshold": 0.0,
    "chunk_score_threshold": 0.5,
    "max_tokens": 2048,
    "temperature": 0.5,
    "top_p": 0.9,
}

print("=" * 80)
print("💬 CHATBOT VỚI WEBSEARCH")
print("=" * 80)
print(f"\n❓ Câu hỏi: {user_question}\n")

# Bước 1: Websearch và lấy context
print("🔍 Đang tìm kiếm thông tin trên web...")
web_retriever = ws_pipeline.retriever.web_retriever
web_sources, rag_sources = await web_retriever.retrive(user_question, generation_params)

print(f"✅ Tìm thấy {len(web_sources)} nguồn web và {len(rag_sources)} chunks")

# Bước 2: Format context
from data_retriever import SourceFormat
from instruction import READER_TEMPLATE, READER_INSTRUCTION

context = SourceFormat()(rag_sources)
prompt = READER_TEMPLATE.format(context=context, question=user_question)

# Bước 3: Gọi LLM để trả lời
print(f"\n🤖 Đang tạo câu trả lời với Qwen 4B LoRA...")
print("-" * 80)
print("\n📝 Câu trả lời:\n")

answer = ""
async for chunk in await vllm_model(
    call_type=CallType.READER,
    instruction=READER_INSTRUCTION,
    prompt=prompt,
    params=generation_params
):
    answer += chunk
    print(chunk, end="", flush=True)

print("\n\n" + "=" * 80)
print("✅ HOÀN THÀNH")
print("=" * 80)

# Lưu kết quả để xem lại
final_result = {
    "question": user_question,
    "answer": answer,
    "web_sources": web_sources,
    "rag_sources": rag_sources
}


###### Note 
If the instruction is too long -> freeze. Seem like vLLM cache instruction, but we still don't know why it only freeze after some requests. (Seem like cache problem, use llm serve would cause it)
